# 00b. Colab에서 aihubshell로 AI Hub 데이터 받기

참고: [코랩에서 AIHUB 데이터 가져오기 (velog, 2024-07-08)](https://velog.io/@river98/%EC%BD%94%EB%9E%A9%EC%97%90%EC%84%9C-AIHUB-%EB%8D%B0%EC%9D%B4%ED%84%B0-%EA%B0%80%EC%A0%B8%EC%98%A4%EA%B8%B0)의 순서를 따른다.

1. AI Hub 공식 `aihubshell`을 Colab에 설치
2. 파일 목록 확인 (`-mode l`)
3. filekey를 하나씩 받아 Colab 디스크(`/content`)에 풀고, 완성된 zip을 Drive로 옮김

블로그와 다른 점:
- 지금 aihubshell(v0.6)은 ID·비밀번호 대신 **API 키**(`-aihubapikey`)를 쓴다. 키는 🔑 Secrets의 `AIHUB_APIKEY` 또는 입력칸으로 받고 셀에 쓰지 않는다 (공개 저장소).
- aihubshell이 받은 결과가 이미 zip이므로 다시 압축하지 않고 Drive로 옮기기만 한다.

> ⚠️ 2026-11-23 Colab에서 같은 AI Hub 다운로드 주소가 `HTTP 502: AI 허브는 해외에서의 데이터 다운로드를 제한하고 있습니다`를 돌려줬다. Colab 서버가 해외에 있기 때문이다. 이 메시지가 나오면 이 노트북으로는 받을 수 없으므로 국내 PC에서 `python -m src.aihub_download download ...`를 실행한다. 제한을 우회(VPN 등)하지 않는다 (AI Hub 이용정책의 국외 이용 조항, docs/DATA_SOURCES.md).

In [ ]:
from google.colab import drive, userdata
import os, subprocess, json, shutil, glob
drive.mount('/content/drive')

try:   # 🔑 Secrets에 AIHUB_APIKEY가 있으면 사용
    AIHUB_APIKEY = userdata.get('AIHUB_APIKEY').strip()
except Exception:   # 없으면 입력칸 (화면에 보이지 않고 노트북에 저장되지 않음)
    from getpass import getpass
    AIHUB_APIKEY = getpass('AI Hub API 키를 붙여 넣고 Enter: ').strip()

DATASET_KEY = '71385'
RAW_DIR = '/content/drive/MyDrive/ML_TeamWork/data/raw/aihub_71385'   # Drive 보관 위치
WORK = '/content/aihub_dl'                                            # Colab 디스크 (임시)
os.makedirs(RAW_DIR, exist_ok=True); os.makedirs(WORK, exist_ok=True)

# 1) aihubshell 설치 (블로그 1~2단계)
!curl -s -o /content/aihubshell https://api.aihub.or.kr/api/aihubshell.do && chmod +x /content/aihubshell
!head -3 /content/aihubshell | tail -1
!df -h /content | tail -1

In [ ]:
# 2) 파일 목록: 파일명 | 용량 | filekey (블로그 '-mode l' 단계)
!cd {WORK} && /content/aihubshell -mode l {DATASET_KEY} | tail -n +10 | head -100

In [ ]:
# 3) 받을 filekey. 저장소 설정 파일(configs/aihub_71385.json)의 preset을 쓴다.
CFG = '/content/drive/MyDrive/ML_TeamWork/26-2-1/configs/aihub_71385.json'
presets = json.load(open(CFG, encoding='utf-8'))['presets']
SELECT = ['smoke_test']            # 먼저 240KB 파일로 확인 → 되면 ['indoor_train_img', 'indoor_val_img']
filekeys = [k for p in SELECT for k in presets[p]['filekeys']]
print(len(filekeys), 'files:', filekeys)

In [ ]:
# 4) 하나씩 받기 → aihubshell이 tar를 풀고 분할 파일을 합침 → 완성된 zip을 Drive로 이동 (블로그 다운로드·저장 단계)
def drive_has(rel):
    return os.path.exists(os.path.join(RAW_DIR, rel))

for fk in filekeys:
    print(f'=== filekey {fk} ===')
    shutil.rmtree(WORK, ignore_errors=True); os.makedirs(WORK)
    r = subprocess.run(['/content/aihubshell', '-mode', 'd', '-datasetkey', DATASET_KEY, '-filekey', str(fk),
                        '-aihubapikey', AIHUB_APIKEY], cwd=WORK, capture_output=True, text=True)
    out = (r.stdout + r.stderr)
    if '해외' in out:
        print('❌ AI Hub가 해외(Colab) IP의 다운로드를 막았습니다. 국내 PC에서 받아야 합니다.')
        print(out[-500:]); break
    zips = glob.glob(f'{WORK}/**/*.zip', recursive=True)
    if not zips:
        print('⚠️ 받은 zip이 없습니다. 출력:'); print(out[-1500:]); break
    for z in zips:
        rel = os.path.relpath(z, WORK)
        if drive_has(rel):
            print('  이미 Drive에 있음:', rel); continue
        os.makedirs(os.path.dirname(os.path.join(RAW_DIR, rel)), exist_ok=True)
        shutil.move(z, os.path.join(RAW_DIR, rel))
        print('  saved', rel, f'{os.path.getsize(os.path.join(RAW_DIR, rel)) / 2**20:,.0f} MB')
shutil.rmtree(WORK, ignore_errors=True)

받은 zip은 풀지 않고 Drive에 둔다. 학습할 때 `src/dataset.py`가 zip 안의 사진을 바로 읽는다 (`01_pipeline_colab.ipynb`).